# Storage Drivers — 02: DuckDB Analytical OTF Queries

**Persona:** Data Analyst

**Goal:** Add `driver:duckdb` as a secondary READ/SEARCH driver backed by a GCS Parquet
file, configure routing so aggregation queries go through DuckDB while writes and
transactional reads stay on PostgreSQL, then run an aggregation.

---

## Prerequisites

- DynaStore running with the `driver:duckdb` extra installed:
  `pip install dynastore[duckdb]`
- A GCS Parquet file is accessible at `GCS_PARQUET_PATH` (e.g. `gs://my-bucket/data.parquet`)
- GCS credentials are available in the environment (`GOOGLE_APPLICATION_CREDENTIALS` or
  Workload Identity) — DuckDB will not silently retry without them
- `driver:postgresql` is already configured on the collection (run notebook 01 first)

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_ADMIN_TOKEN` | _(empty)_ | Bearer token for admin operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Target collection |
| `DUCKDB_POOL_SIZE` | `3` | Max concurrent DuckDB connections |
| `GCS_PARQUET_PATH` | `gs://my-bucket/data.parquet` | GCS path to the Parquet dataset |

In [ ]:
import os
import json
import uuid as _uuid
import time as _t

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL        = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN     = (
    os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or ""
)
# Ephemeral catalog+collection
CATALOG_ID      = f"sd02-{_uuid.uuid4().hex[:8]}"
COLLECTION_ID   = "sentinel2-l2a"
DUCKDB_POOL_SIZE = os.environ.get("DUCKDB_POOL_SIZE", "3")
GCS_PARQUET_PATH = os.environ.get("GCS_PARQUET_PATH", "gs://my-bucket/data.parquet")

_DUCKDB_ENABLED = True  # set False if driver schema is unavailable

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60)

print(f"Base URL         : {BASE_URL}")
print(f"Catalog          : {CATALOG_ID}")
print(f"Collection       : {COLLECTION_ID}")
print(f"DuckDB pool size : {DUCKDB_POOL_SIZE}")
print(f"GCS Parquet path : {GCS_PARQUET_PATH}")
print(f"Auth header      : {'set' if ADMIN_TOKEN else 'not set'}")

In [ ]:
import json as _json

for _attempt in range(3):
    _r = client.post("/stac/catalogs", content=_json.dumps({
        "id": CATALOG_ID, "type": "Catalog", "title": "Storage-Drivers-02 Test",
        "description": "Ephemeral catalog for DuckDB driver demo.", "stac_version": "1.0.0",
    }), headers={"Content-Type": "application/json"})
    if _r.status_code == 201:
        break
    if _attempt < 2:
        print(f"Catalog create attempt {_attempt+1} got {_r.status_code}, retrying in {5*(_attempt+1)}s...")
        _t.sleep(5 * (_attempt + 1))
assert _r.status_code == 201, f"Catalog create failed: {_r.status_code}: {_r.text}"

_r = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk",
    content=_json.dumps({"configs": {
        "CollectionPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {"enabled": True, "operations": {
            "WRITE": [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            "READ":  [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        }},
    }}), headers={"Content-Type": "application/json"}, timeout=60.0)
print(f"Catalog defaults: {_r.status_code}")

_r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections",
    content=_json.dumps({
        "id": COLLECTION_ID, "type": "Collection", "stac_version": "1.0.0",
        "title": "Sentinel-2 L2A Test", "description": "Test.", "license": "proprietary",
        "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]},
                   "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}}, "links": [],
    }), headers={"Content-Type": "application/json"})
assert _r.status_code == 201, f"Collection create failed: {_r.status_code}: {_r.text}"
print(f"Bootstrap: {CATALOG_ID}/{COLLECTION_ID} created")

In [ ]:
resp = client.get("/configs/schemas/CollectionDuckdbDriverConfig")
if resp.status_code != 200:
    print(f"DuckDB driver schema not available ({resp.status_code}) — skipping DuckDB cells.")
    _DUCKDB_ENABLED = False
else:
    _DUCKDB_ENABLED = True
    schema = resp.json()
    properties = schema.get("properties", schema.get("schema", {}).get("properties", {}))

    print("driver:duckdb schema (selected fields):")
    for field in ("enabled", "parquet_path", "pool_size"):
        if field in properties:
            print(f"  {field}: {json.dumps(properties[field], indent=4)}")
        else:
            print(f"  {field}: (not found — check API version or driver extra)")

In [ ]:
if not _DUCKDB_ENABLED:
    print("SKIP: DuckDB driver not available.")
    DUCKDB_CONFIG = {}
else:
    DUCKDB_CONFIG = {
        "enabled": True,
        "parquet_path": GCS_PARQUET_PATH,
        "pool_size": int(DUCKDB_POOL_SIZE),
    }

    resp = client.put(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionDuckdbDriverConfig",
        json=DUCKDB_CONFIG,
    )
    assert resp.status_code == 200, (
        f"Expected 200, got {resp.status_code}: {resp.text[:400]}"
    )

    saved = resp.json()
    print("driver:duckdb config saved:")
    print(json.dumps(saved, indent=2))

In [ ]:
if not _DUCKDB_ENABLED:
    print("SKIP: DuckDB driver not available — routing config skipped.")
    ROUTING_CONFIG = {}
else:
    ROUTING_CONFIG = {
        "enabled": True,
        "operations": {
            "WRITE":  [{"driver_id": "CollectionPostgresqlDriver", "on_failure": "fatal"}],
            "READ":   [{"driver_id": "CollectionPostgresqlDriver", "on_failure": "fatal"}],
            "SEARCH": [{"driver_id": "CollectionDuckdbDriver",     "on_failure": "fatal"}],
        },
    }

    resp = client.put(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
        json=ROUTING_CONFIG,
    )
    assert resp.status_code in (200, 409), (
        f"Expected 200 or 409, got {resp.status_code}: {resp.text[:400]}"
    )

    if resp.status_code == 200:
        print("Routing config saved:")
        print(json.dumps(resp.json(), indent=2))
    else:
        print("Routing already matches (409 Immutable) — no-op.")

In [ ]:
if not _DUCKDB_ENABLED:
    print("SKIP: DuckDB driver not available — aggregation skipped.")
else:
    aggregation_request = {
        "aggregations": [
            {"name": "datetime_frequency", "type": "datetime",
             "property": "properties.datetime", "interval": "1M"},
            {"name": "cloud_cover_stats",  "type": "stats",
             "property": "properties.eo:cloud_cover"},
        ],
    }

    resp = client.post(
        f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/aggregate",
        json=aggregation_request,
    )

    if resp.status_code == 200:
        agg_result = resp.json()
        print("Aggregation result:")
        print(json.dumps(agg_result, indent=2))
    elif resp.status_code == 501:
        print("501 Not Implemented — aggregation extension not enabled on this deployment.")
        print("Enable it by installing dynastore[aggregation] and adding AggregationExtension")
        print("to the application's extension list.")
    else:
        assert False, f"Unexpected status {resp.status_code}: {resp.text[:400]}"

## Edge cases

### Case A — DuckDB in the WRITE routing slot

`driver:duckdb` does not have the `TRANSACTIONS` capability. The platform's capability
gate should reject any routing config that places it in a WRITE slot.
The cell below shows what such a config looks like and the expected rejection.

In [ ]:
if not _DUCKDB_ENABLED:
    print("SKIP: DuckDB driver not available.")
else:
    invalid_routing = {
        "enabled": True,
        "operations": {
            "WRITE": [{"driver_id": "CollectionDuckdbDriver",     "on_failure": "fatal"}],
            "READ":  [{"driver_id": "CollectionPostgresqlDriver", "on_failure": "fatal"}],
        },
    }

    resp = client.put(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
        json=invalid_routing,
    )

    print(f"Invalid routing (DuckDB in WRITE) → {resp.status_code}")
    if resp.status_code in (400, 409, 422):
        detail = resp.json().get("detail", resp.text[:300])
        print(f"  Rejected as expected: {detail}")
    elif resp.status_code == 200:
        print("  WARNING: config was accepted — capability gate may be enforced at write time.")
        print("  Attempting a write will fail with a TRANSACTIONS capability error.")
        client.put(
            f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
            json=ROUTING_CONFIG,
        )
    else:
        print(f"  Unexpected status: {resp.text[:300]}")

In [ ]:
# GCS path without credentials — expected explicit error, not a silent hang
#
# If GOOGLE_APPLICATION_CREDENTIALS is not set and the DuckDB httpfs extension
# cannot resolve credentials from the environment (Workload Identity, etc.),
# DuckDB will raise an IOException when the Parquet scan is opened.
# DynaStore surfaces this as a 500 or 503 with an error message containing
# "GCS" or "credentials" — it will NOT hang waiting for a retry.
#
# To reproduce: temporarily unset GOOGLE_APPLICATION_CREDENTIALS and run a SEARCH.
# The error should appear within the request timeout (default 30 s).
print("GCS credential error behavior (documentation only — not executed):")
print()
print("  Symptom : POST /aggregate returns 500 within 30 s")
print("  Message : 'IOException: failed to open GCS file <path>: ...'")
print("  Fix     : Set GOOGLE_APPLICATION_CREDENTIALS or run on a GCP VM")
print("            with Workload Identity attached to the service account.")
print()
print("  A silent hang (no response for minutes) indicates the httpfs extension")
print("  is attempting a credential refresh loop — upgrade duckdb-httpfs.")

## Teardown

Remove the `driver:duckdb` config. Routing is left pointing to PostgreSQL for
READ/WRITE; the SEARCH slot will fall back to the platform default.

In [ ]:
if _DUCKDB_ENABLED:
    resp = client.delete(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionDuckdbDriverConfig"
    )
    print(f"Delete DuckDB config → {resp.status_code}")
    assert resp.status_code in (204, 404), (
        f"Expected 204/404, got {resp.status_code}: {resp.text[:400]}"
    )

_rd = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
print(f"Teardown: DELETE /stac/catalogs/{CATALOG_ID} → {_rd.status_code}")
assert _rd.status_code == 204, f"Catalog delete failed: {_rd.status_code}: {_rd.text}"

client.close()